In [2]:
import importlib.util, pathlib


def dir_size_mb(path: pathlib.Path) -> float:
    total = sum(f.stat().st_size for f in path.rglob("*") if f.is_file())
    return total / (1024 * 1024)


def pkg_path(name: str):
    spec = importlib.util.find_spec(name)
    if spec is None or not spec.submodule_search_locations:
        return None
    return pathlib.Path(list(spec.submodule_search_locations)[0])


def report(name: str):
    p = pkg_path(name)
    if p is None:
        print(f"  {name:16s}  not installed")
        return
    print(f"  {name:16s}  {dir_size_mb(p):8.1f} MB   ({p})")
    # Largest native libs (Flex/TF ops live in these big pywrap binaries)
    bins = sorted(
        (f for f in p.rglob("*") if f.suffix in {".so", ".pyd", ".dll"} and f.is_file()),
        key=lambda f: f.stat().st_size, reverse=True,
    )[:3]
    for b in bins:
        print(f"      {b.stat().st_size / (1024*1024):8.1f} MB   {b.name}")


if __name__ == "__main__":
    print("Deployment runtime footprints (installed package sizes):")
    for name in ("tensorflow", "tensorflow_cpu", "tflite_runtime", "ai_edge_litert"):
        report(name)

    # sanity check: dir_size_mb sums file sizes correctly (cheap, notebook-safe)
    import tempfile
    with tempfile.TemporaryDirectory() as _d:
        (pathlib.Path(_d) / "probe").write_bytes(b"x" * 1024)
        assert 0 < dir_size_mb(pathlib.Path(_d)) < 1

Deployment runtime footprints (installed package sizes):
  tensorflow          1382.5 MB   (c:\Users\Nicholas Ho\Documents\Programming\Project\Signlingo 2.0\venv\Lib\site-packages\tensorflow)
         920.8 MB   _pywrap_tensorflow_internal.pyd
          12.0 MB   _pywrap_profiler_plugin.pyd
          11.9 MB   stablehlo_extension.pyd
  tensorflow_cpu    not installed
  tflite_runtime    not installed
  ai_edge_litert    not installed
